# DE-BIAS Tag Check


Upload the CSV file with annotations (e.g. `5_ukrainian-folkart-annotations.csv`).


In [4]:
from google.colab import files

files.upload()

Saving 5_ukrainian-folkart-annotations.csv to 5_ukrainian-folkart-annotations (1).csv


{'5_ukrainian-folkart-annotations (1).csv': b'created,value,europeana_id,upvotes,downvotes,recommendation\n13.02.2026,cross,HMT1,7,3,accept\n29.03.2026,Green background,HMT1,4,0,accept\n13.02.2026,mary,HMT1,10,1,accept\n13.02.2026,icon,HMT1,14,0,accept\n27.03.2026,pair of icons,HMT1,9,0,accept\n27.03.2026,Virgin Mary,HMT1,10,0,accept\n27.03.2026,White lily,HMT1,9,0,accept\n13.02.2026,religious icon,HMT1,13,0,accept\n13.02.2026,wedding couple,HMT1,12,2,accept\n29.03.2026,Wooden frame,HMT1,4,0,accept\n29.03.2026,Decorative flowers,HMT1,4,0,accept\n27.03.2026,Blue veil,HMT1,11,0,accept\n13.02.2026,heart,HMT1,11,0,accept\n13.02.2026,couple,HMT1,8,1,accept\n13.02.2026,decorative wooden border,HMT1,11,0,accept\n13.02.2026,halos,HMT1,11,0,accept\n13.02.2026,traditional iconographic style,HMT1,6,2,accept\n13.02.2026,sunburst design,HMT1,2,6,reject\n13.02.2026,geometric patterns,HMT1,13,0,accept\n27.03.2026,Jesus Christ,HMT1,12,0,accept\n27.03.2026,Sacred heart,HMT1,12,0,accept\n13.02.2026,jesu

Upload the RDF vocabulary file (e.g. `DE-BIAS_vocabulary.rdf`).


In [2]:
files.upload()

Saving DE-BIAS_vocabulary.rdf to DE-BIAS_vocabulary.rdf


{'DE-BIAS_vocabulary.rdf': b'<?xml version="1.0" encoding="UTF-8"?>\n<rdf:RDF\n\txmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#">\n\n<rdf:Description rdf:about="http://data.europa.eu/c4p/data/c_63_de">\n\t<rdf:type rdf:resource="http://www.w3.org/2004/02/skos/core#Concept"/>\n\t<hasBiasCategory xmlns="http://data.europa.eu/c4p/ontology#" rdf:resource="http://data.europa.eu/c4p/data/cat_5"/>\n\t<hasContentiousTerm xmlns="http://data.europa.eu/c4p/ontology#" rdf:resource="http://data.europa.eu/c4p/data/t_94_de"/>\n\t<description xmlns="http://purl.org/dc/terms/" xml:lang="de">Behinderte Menschen als \xe2\x80\x9cPflegefall\xe2\x80\x9d zu bezeichnen reduziert sie auf Pflegebed\xc3\xbcrftigkeit. Wenn Menschen zu \xe2\x80\x9cF\xc3\xa4llen\xe2\x80\x9d werden, werden sie als Objekte und Last f\xc3\xbcr die Allgemeinheit wahrgenommen. Sogenannte \xe2\x80\x9cPflegef\xc3\xa4lle\xe2\x80\x9d bekommen vielleicht auch Pers\xc3\xb6nliche Assistenz: Eine Form der allt\xc3\xa4glichen Unterst\xc3

Check annotation tags against DE-BIAS terms.

In [8]:
import csv
import re
import unicodedata
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd

CSV_PATH = Path('5_ukrainian-folkart-annotations.csv')
RDF_PATH = Path('DE-BIAS_vocabulary.rdf')

if not CSV_PATH.exists() or not RDF_PATH.exists():
    raise FileNotFoundError('Upload both files first.')


def norm(s: str) -> str:
    s = unicodedata.normalize('NFKC', s).casefold().strip()
    s = re.sub(r'[^\w\s-]', ' ', s, flags=re.UNICODE)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


root = ET.parse(RDF_PATH).getroot()
rdf = '{http://www.w3.org/1999/02/22-rdf-syntax-ns#}'
lang_key = '{http://www.w3.org/XML/1998/namespace}lang'

terms = {
    norm(literal.text)
    for desc in root.findall(f'{rdf}Description')
    if any(c.tag.endswith('hasContentiousIssue') for c in desc)
    for literal in [next((c for c in desc if c.tag.endswith('literalForm') and (c.text or '').strip()), None)]
    for ambiguous in [next((c for c in desc if c.tag.endswith('isAmbiguous')), None)]
    if (
        literal is not None
        and literal.attrib.get(lang_key) == 'en'
        and ambiguous is not None
        and (ambiguous.text or '').strip().lower() == 'false'
    )
}

with CSV_PATH.open(newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

hits = [
    row
    for row in rows
    if (row.get('value') or '').strip() and norm((row.get('value') or '').strip()) in terms
]

print('=== DE-BIAS Check Result ===')
print(f'Total annotation rows: {len(rows)}')
print(f'DE-BIAS terms used: {len(terms)} (English, non-ambiguous)')
print(f'Flagged rows: {len(hits)}')

if hits:
    df_hits = pd.DataFrame(hits)
    print('\nUnique flagged values:')
    print(', '.join(sorted(df_hits['value'].dropna().unique())))
    print('\nPreview (first 20 flagged rows):')
    display(df_hits.head(20))
else:
    print('\nNo flagged rows found.')


=== DE-BIAS Check Result ===
Total annotation rows: 5946
DE-BIAS terms used: 139 (English, non-ambiguous)
Flagged rows: 1

Unique flagged values:
slave

Preview (first 20 flagged rows):


,created,value,europeana_id,upvotes,downvotes,recommendation
0,13.02.2026,slave,KYD1855,6,3,accept
